In [6]:

# ============================================================
# IEDB Human HLA-I Reference Dataset
# Upload ZIP -> Filter -> Deduplicate -> Random Sample
# Sample size = 229 | Seed = 42
# ============================================================

import os
import re
import csv
import json
import random
import hashlib
import zipfile
from collections import Counter, defaultdict

import pandas as pd
from google.colab import files

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

N = 229
SEED = 42
CHUNK_SIZE = 25000
MIN_LEN = 8
MAX_LEN = 15
OUT = "/content/IEDB_HLAI_outputs"
os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------
# 1. UPLOAD THE COMPRESSED ZIP
# ------------------------------------------------------------

print("Upload mhc_ligand_full_single_file.zip")
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one ZIP file.")

filename = next(iter(uploaded))
zip_path = "/content/" + filename

# If files.upload() supplied the bytes, Colab has also
# written the uploaded file into /content.
del uploaded

if not zipfile.is_zipfile(zip_path):
    raise ValueError("File is not a valid ZIP archive.")

print("ZIP size:",
      round(os.path.getsize(zip_path) / 1024**2, 2),
      "MiB")

# Source checksum for reproducibility
sha = hashlib.sha256()
with open(zip_path, "rb") as handle:
    for block in iter(lambda: handle.read(8 * 1024**2), b""):
        sha.update(block)

source_hash = sha.hexdigest()

# ------------------------------------------------------------
# 2. COLUMN HELPERS
# ------------------------------------------------------------

def clean_name(x):
    return re.sub(r"\s+", " ", str(x)).strip().lower()

def column(columns, group, name, required=True):
    matches = [
        c for c in columns
        if clean_name(c[0]) == clean_name(group)
        and clean_name(c[1]) == clean_name(name)
    ]
    if len(matches) == 1:
        return matches[0]
    if required:
        raise ValueError(
            f"Column missing/ambiguous: {group} / {name}"
        )
    return None

def get_text(df, col):
    if col is None:
        return pd.Series("", index=df.index, dtype=str)
    return df[col].fillna("").astype(str).str.strip()

# ------------------------------------------------------------
# 3. COUNTERS AND METADATA
# ------------------------------------------------------------

counts = Counter()
assay_methods = Counter()
allele_counts = Counter()
length_counts = Counter()

# Unique sequence -> associated experimental metadata
eligible = defaultdict(lambda: {
    "epitope_ids": set(),
    "assay_ids": set(),
    "alleles": set(),
    "methods": set(),
    "responses": set(),
    "outcomes": set(),
    "rows": 0
})

# ------------------------------------------------------------
# 4. READ CSV DIRECTLY INSIDE ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(zip_path) as zf:

    csv_members = [
        x for x in zf.namelist()
        if x.lower().endswith((".csv", ".tsv"))
        and not x.startswith("__MACOSX")
    ]

    if not csv_members:
        raise ValueError("No CSV found inside ZIP.")

    print("Data members:", csv_members)

    for member in csv_members:

        sep = "\t" if member.lower().endswith(".tsv") else ","

        with zf.open(member) as handle:

            reader = pd.read_csv(
                handle,
                sep=sep,
                header=[0, 1],
                dtype=str,
                chunksize=CHUNK_SIZE,
                encoding="utf-8-sig",
                encoding_errors="replace",
                keep_default_na=False,
                low_memory=False,
                on_bad_lines="error"
            )

            for chunk in reader:

                chunk.columns = pd.MultiIndex.from_tuples([
                    (str(a).strip(), str(b).strip())
                    for a, b in chunk.columns
                ])

                cols = chunk.columns

                seq_c = column(cols, "Epitope", "Name")
                type_c = column(cols, "Epitope", "Object Type")
                class_c = column(
                    cols, "MHC Restriction", "Class"
                )
                allele_c = column(
                    cols, "MHC Restriction", "Name"
                )
                outcome_c = column(
                    cols, "Assay", "Qualitative Measurement"
                )
                method_c = column(cols, "Assay", "Method")
                response_c = column(
                    cols, "Assay", "Response measured"
                )

                mod_c = column(
                    cols, "Epitope", "Modified residues"
                )
                mods_c = column(
                    cols, "Epitope", "Modifications"
                )
                epiid_c = column(
                    cols, "Epitope", "Epitope IRI"
                )
                assayid_c = column(
                    cols, "Assay ID", "IEDB IRI",
                    required=False
                )

                seq = get_text(chunk, seq_c).str.upper()
                typ = get_text(chunk, type_c)
                cls = get_text(chunk, class_c)
                allele = get_text(chunk, allele_c)
                outcome = get_text(chunk, outcome_c)
                method = get_text(chunk, method_c)
                response = get_text(chunk, response_c)
                mod = get_text(chunk, mod_c)
                mods = get_text(chunk, mods_c)
                epiid = get_text(chunk, epiid_c)
                assayid = get_text(chunk, assayid_c)

                counts["01_all_rows"] += len(chunk)

                # Linear peptide
                mask = typ.str.casefold().eq("linear peptide")
                counts["02_linear"] += int(mask.sum())

                # Class I
                mask &= cls.str.upper().eq("I")
                counts["03_class_I"] += int(mask.sum())

                # Human classical HLA-I alleles
                mask &= allele.str.match(
                    r"^HLA-[ABC](?=$|[\s*0-9:])",
                    case=False,
                    na=False
                )
                counts["04_HLA_A_B_C"] += int(mask.sum())

                # Positive experimental results
                mask &= outcome.str.match(
                    r"^Positive(?:-|$)",
                    case=False,
                    na=False
                )
                counts["05_positive"] += int(mask.sum())

                # Keep method + response audit BEFORE
                # making a direct-binding classification.
                method_counts = pd.DataFrame({
                    "method": method.loc[mask],
                    "response": response.loc[mask]
                }).value_counts()

                for (m, r), count in method_counts.items():
                    assay_methods[(m, r)] += int(count)

                # Provisional direct-binding definition.
                # Must be checked using the method audit.
                is_direct = method.str.contains(
                    r"(?:^|/)direct(?:/|$)",
                    case=False, regex=True, na=False
                )
                is_mhc = method.str.contains(
                    "mhc", case=False, na=False
                )
                is_binding = response.str.contains(
                    r"binding|affinity|dissociation constant|"
                    r"association constant|\bkd\b|\bki\b|"
                    r"\bic50\b|\bec50\b",
                    case=False, regex=True, na=False
                )

                mask &= is_direct & is_mhc & is_binding
                counts["06_direct_binding"] += int(mask.sum())

                # Length 8-15
                mask &= seq.str.len().between(MIN_LEN, MAX_LEN)
                counts["07_length_8_15"] += int(mask.sum())

                # Only the 20 standard amino-acid letters
                mask &= seq.str.fullmatch(
                    r"[ACDEFGHIKLMNPQRSTVWY]+",
                    na=False
                )
                counts["08_canonical"] += int(mask.sum())

                # No recorded chemical modifications
                mask &= mod.eq("") & mods.eq("")
                counts["09_unmodified"] += int(mask.sum())

                # Collect eligible sequences and provenance
                selected_index = chunk.index[mask]

                for i in selected_index:
                    s = seq.at[i]
                    info = eligible[s]
                    info["rows"] += 1

                    if epiid.at[i]:
                        info["epitope_ids"].add(epiid.at[i])
                    if assayid.at[i]:
                        info["assay_ids"].add(assayid.at[i])

                    info["alleles"].add(allele.at[i])
                    info["methods"].add(method.at[i])
                    info["responses"].add(response.at[i])
                    info["outcomes"].add(outcome.at[i])

                    allele_counts[allele.at[i]] += 1
                    length_counts[len(s)] += 1

                if counts["01_all_rows"] % 500000 < CHUNK_SIZE:
                    print(
                        f"Processed {counts['01_all_rows']:,} rows | "
                        f"Unique eligible: {len(eligible):,}"
                    )

# ------------------------------------------------------------
# 5. UNIQUE SEQUENCES + RANDOM SAMPLE
# ------------------------------------------------------------

all_sequences = sorted(eligible.keys())
counts["10_unique_sequences"] = len(all_sequences)

if len(all_sequences) < N:
    raise ValueError(
        f"Only {len(all_sequences)} eligible sequences. "
        f"Cannot sample {N} without replacement."
    )

rng = random.Random(SEED)
sampled = sorted(rng.sample(all_sequences, N))
counts["11_random_sample"] = len(sampled)

# ------------------------------------------------------------
# 6. SAVE FASTA
# ------------------------------------------------------------

def write_fasta(seqs, path, prefix):
    with open(path, "w", encoding="utf-8") as f:
        for i, s in enumerate(seqs, 1):
            f.write(f">{prefix}_{i:07d}\n{s}\n")

write_fasta(
    all_sequences,
    OUT + "/IEDB_all_eligible_unique.fasta",
    "IEDB_HLAI"
)
write_fasta(
    sampled,
    OUT + "/IEDB_random_229_seed42.fasta",
    "IEDB_RANDOM"
)

# ------------------------------------------------------------
# 7. SAVE METADATA CSV
# ------------------------------------------------------------

def save_metadata(seqs, path):
    with open(path, "w", newline="",
              encoding="utf-8-sig") as f:
        w = csv.writer(f)

        w.writerow([
            "Sequence", "Length",
            "Positive_Direct_Binding_Assay_Rows",
            "Epitope_IRIs", "Assay_IRIs",
            "HLA_Alleles", "Assay_Methods",
            "Responses_Measured", "Outcomes"
        ])

        for s in seqs:
            d = eligible[s]
            w.writerow([
                s, len(s), d["rows"],
                "; ".join(sorted(d["epitope_ids"])),
                "; ".join(sorted(d["assay_ids"])),
                "; ".join(sorted(d["alleles"])),
                "; ".join(sorted(d["methods"])),
                "; ".join(sorted(d["responses"])),
                "; ".join(sorted(d["outcomes"]))
            ])

save_metadata(
    all_sequences,
    OUT + "/IEDB_all_eligible_metadata.csv"
)
save_metadata(
    sampled,
    OUT + "/IEDB_random_229_seed42_metadata.csv"
)

# ------------------------------------------------------------
# 8. REPORTS
# ------------------------------------------------------------

pd.DataFrame([
    {"Step": key, "Count": value}
    for key, value in sorted(counts.items())
]).to_csv(OUT + "/filter_report.csv", index=False)

def binding_rule(m, r):
    return (
        bool(re.search(r"(?:^|/)direct(?:/|$)", m, re.I))
        and bool(re.search("mhc", m, re.I))
        and bool(re.search(
            r"binding|affinity|dissociation constant|"
            r"association constant|\bkd\b|\bki\b|"
            r"\bic50\b|\bec50\b", r, re.I
        ))
    )

pd.DataFrame([
    {
        "Assay_Method": m,
        "Response_Measured": r,
        "Rows": count,
        "Accepted_by_rule": binding_rule(m, r)
    }
    for (m, r), count in assay_methods.most_common()
]).to_csv(OUT + "/assay_method_audit.csv", index=False)

pd.DataFrame([
    {"Allele": a, "Eligible_Assay_Rows": n}
    for a, n in allele_counts.most_common()
]).to_csv(OUT + "/allele_counts.csv", index=False)

pd.DataFrame([
    {"Length": length, "Eligible_Assay_Rows": n}
    for length, n in sorted(length_counts.items())
]).to_csv(OUT + "/length_counts.csv", index=False)

with open(OUT + "/methods_metadata.json", "w") as f:
    json.dump({
        "source_filename": filename,
        "source_zip_sha256": source_hash,
        "source_type": "IEDB MHC Ligand database CSV export",
        "sample_size": N,
        "seed": SEED,
        "sampling": "Python random.Random.sample",
        "sampling_frame": "sorted unique eligible sequences",
        "replacement": False,
        "peptide_length_range": [MIN_LEN, MAX_LEN],
        "assay_rule": "provisional direct-binding classification; audit required",
        "deduplication": "exact sequence"
    }, f, indent=2)

# ------------------------------------------------------------
# 9. ZIP OUTPUTS
# ------------------------------------------------------------

print("\n========== FILTER REPORT ==========")
for key, value in sorted(counts.items()):
    print(f"{key}: {value:,}")

print("\nTop assay method/response combinations:")
for (m, r), n in assay_methods.most_common(12):
    print(f"{n:,} | {m} | {r}")

output_zip = "/content/IEDB_HLAI_229_Seed42_results.zip"

with zipfile.ZipFile(
    output_zip, "w", zipfile.ZIP_DEFLATED
) as zf:
    for name in os.listdir(OUT):
        zf.write(os.path.join(OUT, name), arcname=name)

print("\nDownload ready:", output_zip)
files.download(output_zip)


Upload mhc_ligand_full_single_file.zip


Saving mhc_ligand_full_single_file.zip to mhc_ligand_full_single_file (1).zip
ZIP size: 308.35 MiB
Data members: ['mhc_ligand_full.csv']
Processed 500,000 rows | Unique eligible: 8,864
Processed 1,000,000 rows | Unique eligible: 9,162
Processed 1,500,000 rows | Unique eligible: 9,461
Processed 2,000,000 rows | Unique eligible: 9,619
Processed 2,500,000 rows | Unique eligible: 9,619
Processed 3,000,000 rows | Unique eligible: 9,619
Processed 3,500,000 rows | Unique eligible: 10,151
Processed 4,000,000 rows | Unique eligible: 10,154
Processed 4,500,000 rows | Unique eligible: 10,355
Processed 5,000,000 rows | Unique eligible: 13,115
Processed 5,500,000 rows | Unique eligible: 13,208
Processed 6,000,000 rows | Unique eligible: 13,409

========== FILTER REPORT ==========
01_all_rows: 6,275,428
02_linear: 6,274,657
03_class_I: 4,280,017
04_HLA_A_B_C: 1,668,697
05_positive: 1,553,210
06_direct_binding: 22,785
07_length_8_15: 22,580
08_canonical: 22,574
09_unmodified: 22,574
10_unique_sequenc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>